In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session




# ============================================================
# NOTEBOOK A (ResNet-18 only) — COPY/PASTE READY
# - Datasets: PCam (+ optional PANDA if available)
# - Methods: SimCLR, BYOL
# - Backbone: ResNet-18
# - Protocol: 3 splits × 3 seeds (9 runs)
# - Label-efficiency: 1%, 5%, 10% (stratified)
# - Metrics: Acc, AUROC, F1, ECE, Brier
#
# CRITICAL FIX INCLUDED:
# ✅ Feature standardization fitted on TRAIN only (mu/std), applied to VAL with same params.
#    (No more train/val separate standardization bug.)
#
# EXTRA SAFETY:
# ✅ Incremental checkpoint to /kaggle/working/_partial_results_raw.csv after each label_frac eval.
# ============================================================

import os
import time
import random
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from PIL import Image

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss

import h5py

# -----------------------
# 0) CONFIG
# -----------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory/1024**3:.2f} GB")

# Protocol (ResNet)
N_SPLITS = 3
SEEDS = [0, 1, 2]
SPLIT_BASE_SEED = 1000
LABEL_FRACS = [0.01, 0.05, 0.10]
VAL_RATIO = 0.2

# Training
SSL_EPOCHS = 10
PROBE_EPOCHS = 10
LR_SSL = 1e-3
LR_PROBE = 1e-4

# Supervised sanity baselines
SUP_EPOCHS = 10
LR_SUP = 1e-3
WD_SUP = 1e-4

# Debug subsampling (set to 1.0 for full)
SAMPLE_FRAC_PCAM = 0.2
SAMPLE_FRAC_PANDA = 0.2

# Batch sizes
BATCH_SSL_RESNET = 128
BATCH_SUP_RESNET = 128

# Image sizes
IMG_PCAM_RESNET = 96
IMG_PANDA = 224

# Workers
NUM_WORKERS_PATH = 4
NUM_WORKERS_H5 = 0
PIN_MEMORY = True

# AMP
USE_AMP = (DEVICE.type == "cuda")

# Checkpoints
PARTIAL_RAW_PATH = "/kaggle/working/_partial_results_raw.csv"

# -----------------------
# 1) REPRODUCIBILITY
# -----------------------
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# -----------------------
# 2) HELPER: FIND FILES UNDER /kaggle/input
# -----------------------
def find_any_under_input(pattern: str) -> List[Path]:
    base = Path("/kaggle/input")
    return list(base.rglob(pattern))

def pick_shortest(paths: List[Path]) -> Optional[Path]:
    if not paths:
        return None
    return sorted(paths, key=lambda p: len(str(p)))[0]

# -----------------------
# 3) METRICS + SAFETY
# -----------------------
def sanitize_np_probs(probs: np.ndarray) -> np.ndarray:
    probs = np.asarray(probs).reshape(-1)
    probs = np.nan_to_num(probs, nan=0.5, posinf=1.0, neginf=0.0)
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    return probs

def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    probs = sanitize_np_probs(probs)
    labels = np.asarray(labels).reshape(-1).astype(int)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (probs >= lo) & (probs < hi)
        if m.sum() == 0:
            continue
        bin_conf = probs[m].mean()
        bin_acc = labels[m].mean()
        ece += (m.sum() / n) * abs(bin_acc - bin_conf)
    return float(ece)

def compute_metrics_binary(probs: np.ndarray, labels: np.ndarray, thr: float = 0.5) -> Dict[str, float]:
    probs = sanitize_np_probs(probs)
    labels = np.asarray(labels).reshape(-1).astype(int)
    preds = (probs >= thr).astype(int)

    acc = float((preds == labels).mean())
    try:
        auc = float(roc_auc_score(labels, probs))
    except ValueError:
        auc = float("nan")
    f1 = float(f1_score(labels, preds, zero_division=0))
    brier = float(brier_score_loss(labels, probs))
    ece = compute_ece(probs, labels, n_bins=15)
    return {"acc": acc, "auc": auc, "f1": f1, "ece": ece, "brier": brier}

def sanitize_tensor(x: torch.Tensor, clamp: float = 1e4) -> torch.Tensor:
    x = x.float()
    x = torch.nan_to_num(x, nan=0.0, posinf=clamp, neginf=-clamp)
    x = torch.clamp(x, -clamp, clamp)
    return x

# -----------------------
# 3.5) STANDARDIZER (CRITICAL FIX)
# -----------------------
@torch.no_grad()
def fit_standardizer(train_feats: torch.Tensor, eps: float = 1e-6) -> Tuple[torch.Tensor, torch.Tensor]:
    """Fit (mu, sd) on TRAIN only."""
    x = train_feats.float()
    mu = x.mean(dim=0, keepdim=True)
    sd = x.std(dim=0, keepdim=True).clamp_min(eps)
    return mu, sd

@torch.no_grad()
def apply_standardizer(feats: torch.Tensor, mu: torch.Tensor, sd: torch.Tensor) -> torch.Tensor:
    """Apply TRAIN-fitted (mu, sd) to any split (train/val/test)."""
    return (feats.float() - mu) / sd

# -----------------------
# 4) DATASETS (Path-based and HDF5-based)
# -----------------------
class TwoViewSSL_Path(Dataset):
    def __init__(self, image_paths: List[str], img_size: int):
        self.image_paths = image_paths
        self.t = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.RandomGrayscale(p=0.1),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5)),
        ])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.t(img), self.t(img)

class Supervised_Path(Dataset):
    def __init__(self, image_paths: List[str], labels: List[int], img_size: int):
        assert len(image_paths) == len(labels)
        self.image_paths = image_paths
        self.labels = labels
        self.t = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5)),
        ])
    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.t(img), int(self.labels[idx])

def infer_h5_keys(h5_path: str) -> Tuple[str, str]:
    img_candidates = ["images", "imgs", "x", "X", "data", "image"]
    lbl_candidates = ["labels", "y", "Y", "targets", "target", "label"]
    with h5py.File(h5_path, "r") as f:
        keys = list(f.keys())

        img_key = next((k for k in img_candidates if k in keys), None)
        lbl_key = next((k for k in lbl_candidates if k in keys), None)
        if img_key is not None and lbl_key is not None:
            return img_key, lbl_key

        img_key2, lbl_key2 = None, None
        for k in keys:
            d = f[k]
            if not hasattr(d, "shape"):
                continue
            shape = d.shape
            if len(shape) >= 3:
                if (len(shape) == 4 and (shape[-1] == 3 or shape[1] == 3)) or (len(shape) == 3 and shape[-1] == 3):
                    img_key2 = k
            if len(shape) == 1:
                lbl_key2 = k

        if img_key2 is None:
            for k in keys:
                d = f[k]
                if hasattr(d, "shape") and len(d.shape) >= 3:
                    img_key2 = k
                    break

        if lbl_key2 is None:
            for k in keys:
                d = f[k]
                if hasattr(d, "shape") and len(d.shape) == 1:
                    lbl_key2 = k
                    break

        if img_key2 is None or lbl_key2 is None:
            raise KeyError(f"Could not infer image/label keys in H5: {h5_path}\nFound keys: {keys}")
        return img_key2, lbl_key2

class _H5HandleMixin:
    def __init__(self, h5_path: str):
        self.h5_path = h5_path
        self._h5 = None
    def _get_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5
    def __del__(self):
        try:
            if self._h5 is not None:
                self._h5.close()
        except Exception:
            pass

class TwoViewSSL_H5(Dataset, _H5HandleMixin):
    def __init__(self, h5_path: str, indices: np.ndarray, img_key: str, img_size: int):
        Dataset.__init__(self)
        _H5HandleMixin.__init__(self, h5_path)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.img_key = img_key
        self.t = transforms.Compose([
            transforms.RandomResizedCrop(img_size, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
            transforms.RandomGrayscale(p=0.1),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5)),
        ])
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        f = self._get_h5()
        i = int(self.indices[idx])
        arr = f[self.img_key][i]
        if arr.ndim == 3 and arr.shape[0] == 3 and arr.shape[-1] != 3:
            arr = np.transpose(arr, (1, 2, 0))
        img = Image.fromarray(arr.astype(np.uint8))
        return self.t(img), self.t(img)

class Supervised_H5(Dataset, _H5HandleMixin):
    def __init__(self, h5_path: str, indices: np.ndarray, img_key: str, labels_full: np.ndarray, img_size: int):
        Dataset.__init__(self)
        _H5HandleMixin.__init__(self, h5_path)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.img_key = img_key
        self.labels_full = np.asarray(labels_full, dtype=np.int64)
        self.t = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5)),
        ])
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        f = self._get_h5()
        i = int(self.indices[idx])
        arr = f[self.img_key][i]
        if arr.ndim == 3 and arr.shape[0] == 3 and arr.shape[-1] != 3:
            arr = np.transpose(arr, (1, 2, 0))
        img = Image.fromarray(arr.astype(np.uint8))
        x = self.t(img)
        y = int(self.labels_full[i])
        return x, y

# -----------------------
# 5) DATA SOURCE ABSTRACTION
# -----------------------
class DataSourceBase:
    def __init__(self, name: str):
        self.name = name
    def __len__(self) -> int:
        raise NotImplementedError
    def labels_all(self) -> np.ndarray:
        raise NotImplementedError
    def make_ssl_dataset(self, indices: np.ndarray, img_size: int) -> Dataset:
        raise NotImplementedError
    def make_sup_dataset(self, indices: np.ndarray, img_size: int) -> Dataset:
        raise NotImplementedError

class PathDataSource(DataSourceBase):
    def __init__(self, name: str, paths: List[str], labels: List[int]):
        super().__init__(name)
        self.paths = list(paths)
        self.labels = list(map(int, labels))
    def __len__(self) -> int:
        return len(self.paths)
    def labels_all(self) -> np.ndarray:
        return np.asarray(self.labels, dtype=np.int64)
    def make_ssl_dataset(self, indices: np.ndarray, img_size: int) -> Dataset:
        sel_paths = [self.paths[i] for i in indices]
        return TwoViewSSL_Path(sel_paths, img_size=img_size)
    def make_sup_dataset(self, indices: np.ndarray, img_size: int) -> Dataset:
        sel_paths = [self.paths[i] for i in indices]
        sel_labels = [self.labels[i] for i in indices]
        return Supervised_Path(sel_paths, sel_labels, img_size=img_size)

class H5DataSource(DataSourceBase):
    def __init__(self, name: str, h5_path: str, img_key: str, labels: np.ndarray):
        super().__init__(name)
        self.h5_path = h5_path
        self.img_key = img_key
        self._labels = np.asarray(labels, dtype=np.int64)
    def __len__(self) -> int:
        return len(self._labels)
    def labels_all(self) -> np.ndarray:
        return self._labels
    def make_ssl_dataset(self, indices: np.ndarray, img_size: int) -> Dataset:
        return TwoViewSSL_H5(self.h5_path, indices, img_key=self.img_key, img_size=img_size)
    def make_sup_dataset(self, indices: np.ndarray, img_size: int) -> Dataset:
        return Supervised_H5(self.h5_path, indices, img_key=self.img_key, labels_full=self._labels, img_size=img_size)

# -----------------------
# 6) PCAM LOADER (AUTO-DETECT)
# -----------------------
def load_pcam_datasource(sample_frac: float = 1.0) -> DataSourceBase:
    train_labels_paths = find_any_under_input("train_labels.csv")
    if train_labels_paths:
        labels_path = str(pick_shortest(train_labels_paths))
        pcam_root = str(Path(labels_path).parent)

        df = pd.read_csv(labels_path)
        if 0 < sample_frac < 1.0:
            df = df.sample(frac=sample_frac, random_state=42).reset_index(drop=True)

        def make_path(img_id: str):
            tif = os.path.join(pcam_root, "train", f"{img_id}.tif")
            if os.path.exists(tif):
                return tif
            for ext in ["png", "jpg", "jpeg"]:
                p = os.path.join(pcam_root, "train", f"{img_id}.{ext}")
                if os.path.exists(p):
                    return p
            return tif

        df["image_path"] = df["id"].astype(str).apply(make_path)
        df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

        ds = PathDataSource("pcam", df["image_path"].tolist(), df["label"].astype(int).tolist())
        ds._universe_indices = np.arange(len(ds), dtype=np.int64)
        return ds

    h5_candidates = find_any_under_input("training_split.h5")
    if h5_candidates:
        h5_path = str(pick_shortest(h5_candidates))
        img_key, lbl_key = infer_h5_keys(h5_path)
        with h5py.File(h5_path, "r") as f:
            y = np.array(f[lbl_key][:], dtype=np.int64)

        ds = H5DataSource("pcam", h5_path, img_key, labels=y)
        if 0 < sample_frac < 1.0:
            sss = StratifiedShuffleSplit(
                n_splits=1,
                train_size=max(1, int(len(y) * sample_frac)),
                random_state=42
            )
            idx_small, _ = next(sss.split(np.zeros(len(y)), y))
            ds._universe_indices = np.asarray(idx_small, dtype=np.int64)
        else:
            ds._universe_indices = np.arange(len(y), dtype=np.int64)
        return ds

    raise FileNotFoundError(
        "PCam not found. Add either:\n"
        "- histopathologic-cancer-detection (train_labels.csv)\n"
        "- or a dataset that contains training_split.h5"
    )

# -----------------------
# 7) PANDA LOADER (AUTO-DETECT)
# -----------------------
def infer_panda_resized_dir() -> Optional[str]:
    candidates = find_any_under_input("train_images/train_images")
    if not candidates:
        return None
    best, best_count = None, -1
    for c in candidates:
        cnt = len(list(c.glob("*.png")))
        if cnt > best_count:
            best, best_count = c, cnt
    return str(best) if best is not None else None

def infer_panda_comp_dir() -> Optional[str]:
    candidates = find_any_under_input("train.csv")
    good = []
    for c in candidates:
        parent = c.parent
        if (parent / "sample_submission.csv").exists():
            txt = str(parent).lower()
            if "prostate" in txt or "panda" in txt:
                good.append(parent)
    if good:
        good = sorted(good, key=lambda p: len(str(p)))
        return str(good[0])
    return None

def load_panda_datasource(sample_frac: float = 1.0) -> Optional[DataSourceBase]:
    resized_dir = infer_panda_resized_dir()
    comp_dir = infer_panda_comp_dir()
    if resized_dir is None or comp_dir is None:
        print("\n[WARN] PANDA skipped (missing resized images folder OR competition train.csv).")
        return None

    train_csv = os.path.join(comp_dir, "train.csv")
    df = pd.read_csv(train_csv)[["image_id", "isup_grade"]].dropna()
    if 0 < sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=42).reset_index(drop=True)

    df["image_path"] = df["image_id"].astype(str).apply(lambda x: os.path.join(resized_dir, f"{x}.png"))
    df = df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)

    labels = [1 if int(g) >= 2 else 0 for g in df["isup_grade"].astype(int).tolist()]
    paths = df["image_path"].tolist()
    ds = PathDataSource("panda", paths, labels)
    ds._universe_indices = np.arange(len(ds), dtype=np.int64)
    return ds

# -----------------------
# 8) SPLITS + STRATIFIED LABEL SUBSET
# -----------------------
def make_splits_indices(universe_indices: np.ndarray, labels: np.ndarray, n_splits: int, val_ratio: float, base_seed: int):
    labels = np.asarray(labels, dtype=np.int64)
    splits = []
    for split_id in range(n_splits):
        sss = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio, random_state=base_seed + split_id)
        tr_rel, va_rel = next(sss.split(np.zeros(len(universe_indices)), labels[universe_indices]))
        splits.append((universe_indices[tr_rel], universe_indices[va_rel]))
    return splits

def stratified_label_subset_indices(train_indices: np.ndarray, labels: np.ndarray, frac: float, seed: int):
    labels = np.asarray(labels, dtype=np.int64)
    n = len(train_indices)
    k = max(1, int(n * frac))
    sss = StratifiedShuffleSplit(n_splits=1, train_size=k, random_state=seed)
    sel_rel, _ = next(sss.split(np.zeros(n), labels[train_indices]))
    return train_indices[sel_rel]

# -----------------------
# 9) BACKBONE: ResNet-18
# -----------------------
def get_resnet18(pretrained: bool = False):
    try:
        from torchvision.models import ResNet18_Weights
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        m = models.resnet18(weights=weights)
    except Exception:
        m = models.resnet18(pretrained=pretrained)
    feat_dim = m.fc.in_features
    m.fc = nn.Identity()
    return m, feat_dim

# -----------------------
# 10) SIMCLR
# -----------------------
class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, proj_dim: int = 128, hidden_dim: int = 2048):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )
    def forward(self, x): return self.net(x)

def get_simclr_encoder_resnet18(proj_dim: int = 128) -> nn.Module:
    backbone, feat_dim = get_resnet18(pretrained=False)
    proj = ProjectionHead(feat_dim, proj_dim=proj_dim, hidden_dim=2048)
    return nn.Sequential(backbone, proj)

def nt_xent_loss_fp32(z1, z2, temperature=0.5):
    z1 = z1.float()
    z2 = z2.float()
    b = z1.size(0)

    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    z = torch.cat([z1, z2], dim=0)
    sim = torch.matmul(z, z.T) / float(temperature)

    mask = torch.eye(2 * b, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(mask, -1e4)

    labels = torch.arange(b, device=z.device)
    labels = torch.cat([labels + b, labels], dim=0)
    return F.cross_entropy(sim, labels)

def pretrain_simclr(encoder: nn.Module, ssl_dataset: Dataset, batch_size: int, epochs: int, lr: float, num_workers: int):
    encoder = encoder.to(DEVICE)
    opt = torch.optim.Adam(encoder.parameters(), lr=lr)
    dl = DataLoader(
        ssl_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=PIN_MEMORY, drop_last=True
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    encoder.train()
    for ep in range(1, epochs + 1):
        t0 = time.time()
        total, n = 0.0, 0
        for x1, x2 in dl:
            x1 = x1.to(DEVICE, non_blocking=True)
            x2 = x2.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                z1 = encoder(x1)
                z2 = encoder(x2)
            loss = nt_xent_loss_fp32(z1, z2, temperature=0.5)
            if not torch.isfinite(loss):
                print("[WARN] SimCLR loss non-finite, skipping batch.")
                continue
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total += float(loss.item()) * x1.size(0)
            n += x1.size(0)
        print(f"[SimCLR] ep {ep}/{epochs} loss={total/max(1,n):.4f} time={time.time()-t0:.1f}s")
    return encoder

# -----------------------
# 11) BYOL (ResNet-18)
# -----------------------
import copy

class BYOL(nn.Module):
    def __init__(self, proj_dim=256, hidden_dim=4096, moving_avg_decay=0.996):
        super().__init__()
        backbone, feat_dim = get_resnet18(pretrained=False)

        self.online_encoder = backbone
        self.online_proj = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )
        self.online_pred = nn.Sequential(
            nn.Linear(proj_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, proj_dim),
        )

        self.target_encoder = copy.deepcopy(self.online_encoder)
        self.target_proj = copy.deepcopy(self.online_proj)
        for p in self.target_encoder.parameters(): p.requires_grad = False
        for p in self.target_proj.parameters(): p.requires_grad = False
        self.m = moving_avg_decay

    @torch.no_grad()
    def update_target(self):
        for op, tp in zip(self.online_encoder.parameters(), self.target_encoder.parameters()):
            tp.data = self.m * tp.data + (1 - self.m) * op.data
        for op, tp in zip(self.online_proj.parameters(), self.target_proj.parameters()):
            tp.data = self.m * tp.data + (1 - self.m) * op.data

    def forward(self, x1, x2):
        y1 = self.online_encoder(x1)
        y2 = self.online_encoder(x2)
        z1 = self.online_proj(y1)
        z2 = self.online_proj(y2)
        p1 = self.online_pred(z1)
        p2 = self.online_pred(z2)
        with torch.no_grad():
            t1 = self.target_proj(self.target_encoder(x1))
            t2 = self.target_proj(self.target_encoder(x2))
        return p1, p2, t1.detach(), t2.detach()

    @staticmethod
    def loss_fn(p, z):
        p = F.normalize(p.float(), dim=1)
        z = F.normalize(z.float(), dim=1)
        return 2 - 2 * (p * z).sum(dim=1).mean()

def pretrain_byol(model: BYOL, ssl_dataset: Dataset, batch_size: int, epochs: int, lr: float, num_workers: int):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    dl = DataLoader(
        ssl_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=PIN_MEMORY, drop_last=True
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    model.train()
    for ep in range(1, epochs + 1):
        t0 = time.time()
        total, n = 0.0, 0
        for x1, x2 in dl:
            x1 = x1.to(DEVICE, non_blocking=True)
            x2 = x2.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                p1, p2, t1, t2 = model(x1, x2)
                loss = BYOL.loss_fn(p1, t2) + BYOL.loss_fn(p2, t1)
            if not torch.isfinite(loss):
                print("[WARN] BYOL loss non-finite, skipping batch.")
                continue
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            model.update_target()
            total += float(loss.item()) * x1.size(0)
            n += x1.size(0)
        print(f"[BYOL]   ep {ep}/{epochs} loss={total/max(1,n):.4f} time={time.time()-t0:.1f}s")
    return model

# -----------------------
# 12) FEATURE EXTRACTION + LINEAR PROBE (with FIXED standardization)
# -----------------------
@torch.no_grad()
def extract_features(backbone: nn.Module, dl: DataLoader) -> Tuple[torch.Tensor, np.ndarray]:
    backbone.eval()
    backbone = backbone.to(DEVICE)
    feats, ys = [], []
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            z = backbone(x)
        if isinstance(z, (tuple, list)):
            z = z[0]
        if z.ndim > 2:
            z = torch.flatten(z, 1)
        z = sanitize_tensor(z)
        feats.append(z.detach().cpu())
        ys.append(y.numpy())
    feats = sanitize_tensor(torch.cat(feats, dim=0))
    ys = np.concatenate(ys, axis=0).astype(int)
    return feats, ys

class LinearProbe(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.fc = nn.Linear(in_dim, 1)
    def forward(self, x):
        return self.fc(x).squeeze(1)

def train_linear_probe_on_standardized_feats(feats_std: torch.Tensor, labels: np.ndarray, epochs: int, lr: float, batch_size: int = 256) -> nn.Module:
    X = sanitize_tensor(feats_std).float()
    y = torch.tensor(labels, dtype=torch.float32)
    ds = torch.utils.data.TensorDataset(X, y)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)

    clf = LinearProbe(X.size(1)).to(DEVICE)
    opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss()

    clf.train()
    for ep in range(1, epochs + 1):
        total, n = 0.0, 0
        for xb, yb in dl:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            logits = sanitize_tensor(clf(xb))
            loss = crit(logits, yb)
            if not torch.isfinite(loss):
                print("[WARN] Probe loss non-finite. Skipping batch.")
                continue
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(clf.parameters(), 1.0)
            opt.step()
            total += float(loss.item()) * xb.size(0)
            n += xb.size(0)
        print(f"[Probe]  ep {ep}/{epochs} loss={total/max(1,n):.4f}")
    return clf

@torch.no_grad()
def eval_linear_probe_on_standardized_feats(clf: nn.Module, feats_std: torch.Tensor, labels: np.ndarray) -> Dict[str, float]:
    clf.eval()
    X = sanitize_tensor(feats_std).float().to(DEVICE)
    logits = sanitize_tensor(clf(X).detach().cpu())
    probs = sanitize_np_probs(torch.sigmoid(logits).numpy())
    return compute_metrics_binary(probs, labels, thr=0.5)


# -----------------------
# 12.5) SUPERVISED SANITY BASELINES (ResNet-18)
# -----------------------

def train_supervised_resnet18(train_dl: DataLoader, val_dl: DataLoader, pretrained: bool, epochs: int, lr: float, wd: float) -> Dict[str, float]:
    """Train end-to-end supervised ResNet-18 and evaluate on val.

    This is a sanity baseline to ensure the pipeline can learn the task with labels.
    """
    backbone, feat_dim = get_resnet18(pretrained=pretrained)
    head = nn.Linear(feat_dim, 1)
    model = nn.Sequential(backbone, head).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    crit = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    for ep in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for xb, yb in train_dl:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.float().to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(xb).squeeze(1)
                loss = crit(logits, yb)

            if not torch.isfinite(loss):
                print('[WARN] Supervised loss non-finite. Skipping batch.')
                continue

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            total += float(loss.item()) * xb.size(0)
            n += xb.size(0)
        print(f'[SUP]    ep {ep}/{epochs} loss={total/max(1,n):.4f}')

    # Evaluate
    model.eval()
    probs_all, y_all = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            xb = xb.to(DEVICE, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(xb).squeeze(1)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            probs_all.append(probs)
            y_all.append(yb.numpy())

    probs_all = sanitize_np_probs(np.concatenate(probs_all, axis=0))
    y_all = np.concatenate(y_all, axis=0).astype(int)
    return compute_metrics_binary(probs_all, y_all, thr=0.5)

# -----------------------
# 13) RUNNER
# -----------------------
def cfg_for(dataset_name: str):
    d = dataset_name.lower()
    if d == "pcam":
        return IMG_PCAM_RESNET, BATCH_SSL_RESNET, BATCH_SUP_RESNET
    if d == "panda":
        return IMG_PANDA, BATCH_SSL_RESNET, BATCH_SUP_RESNET
    raise ValueError(f"Unknown dataset: {dataset_name}")

def append_checkpoint(rows: List[dict]):
    if not rows:
        return
    os.makedirs("/kaggle/working", exist_ok=True)
    pd.DataFrame(rows).to_csv(PARTIAL_RAW_PATH, index=False)

def run_config_resnet18(
    ds: DataSourceBase,
    universe_indices: np.ndarray,
    method: str,
):
    dataset_name = ds.name
    labels_all = ds.labels_all()
    img_size, batch_ssl, batch_sup = cfg_for(dataset_name)
    use_workers = NUM_WORKERS_H5 if isinstance(ds, H5DataSource) else NUM_WORKERS_PATH

    splits = make_splits_indices(
        universe_indices=universe_indices,
        labels=labels_all,
        n_splits=N_SPLITS,
        val_ratio=VAL_RATIO,
        base_seed=SPLIT_BASE_SEED
    )

    rows = []

    for split_id, (tr_idx, va_idx) in enumerate(splits):
        for seed in SEEDS:
            run_seed = 10_000 + 100 * split_id + seed
            set_seed(run_seed)

            print("\n" + "=" * 92)
            print(f"[RUN] dataset={dataset_name} method={method} backbone=resnet18 "
                  f"split={split_id}/{N_SPLITS-1} seed={seed} img={img_size} | n_tr={len(tr_idx)} n_va={len(va_idx)}")
            print("=" * 92)

            ssl_ds = ds.make_ssl_dataset(tr_idx, img_size=img_size)
            method_l = method.lower()

            if method_l == "simclr":
                enc = get_simclr_encoder_resnet18(proj_dim=128)
                enc = pretrain_simclr(enc, ssl_ds, batch_size=batch_ssl, epochs=SSL_EPOCHS, lr=LR_SSL, num_workers=use_workers)
                backbone = enc[0]
            elif method_l == "byol":
                byol = BYOL(proj_dim=256, hidden_dim=4096)
                byol = pretrain_byol(byol, ssl_ds, batch_size=batch_ssl, epochs=SSL_EPOCHS, lr=LR_SSL, num_workers=use_workers)
                backbone = byol.online_encoder
            elif method_l in ["sup_scratch", "sup_imagenet"]:
                backbone = None  # handled per-label-fraction below
            else:
                raise ValueError(f"Unknown method: {method}")

            val_ds = ds.make_sup_dataset(va_idx, img_size=img_size)
            val_dl = DataLoader(val_ds, batch_size=batch_sup, shuffle=False, num_workers=use_workers, pin_memory=PIN_MEMORY)

            for frac in LABEL_FRACS:
                sub_seed = 42 + seed + 10 * split_id
                sub_tr_idx = stratified_label_subset_indices(tr_idx, labels_all, frac=frac, seed=sub_seed)
                print(f"\n  [LABEL] frac={frac*100:.2f}% n_lab={len(sub_tr_idx)} (split={split_id}, seed={seed})")

                train_ds = ds.make_sup_dataset(sub_tr_idx, img_size=img_size)
                train_dl = DataLoader(train_ds, batch_size=batch_sup, shuffle=True, num_workers=use_workers, pin_memory=PIN_MEMORY)

                if method_l in ["sup_scratch", "sup_imagenet"]:
                    # End-to-end supervised sanity baseline
                    metrics = train_supervised_resnet18(
                        train_dl, val_dl,
                        pretrained=(method_l == "sup_imagenet"),
                        epochs=SUP_EPOCHS, lr=LR_SUP, wd=WD_SUP
                    )
                else:
                    # SSL + frozen linear probe
                    # Extract features
                    train_feats, train_y = extract_features(backbone, train_dl)
                    val_feats, val_y = extract_features(backbone, val_dl)

                    # ✅ CRITICAL FIX: fit on TRAIN only, apply to VAL with same mu/sd
                    mu, sd = fit_standardizer(train_feats)
                    train_feats_std = apply_standardizer(train_feats, mu, sd)
                    val_feats_std = apply_standardizer(val_feats, mu, sd)

                    clf = train_linear_probe_on_standardized_feats(train_feats_std, train_y, epochs=PROBE_EPOCHS, lr=LR_PROBE, batch_size=256)
                    metrics = eval_linear_probe_on_standardized_feats(clf, val_feats_std, val_y)

                row = {
                    "dataset": dataset_name,
                    "method": method_l,
                    "backbone": "resnet18",
                    "img_size": img_size,
                    "split_id": split_id,
                    "seed": seed,
                    "label_frac": frac,
                    **metrics
                }
                rows.append(row)

                # incremental checkpoint
                append_checkpoint(rows)

    return pd.DataFrame(rows)

# -----------------------
# 14) MAIN
# -----------------------
os.makedirs("/kaggle/working", exist_ok=True)
if os.path.exists(PARTIAL_RAW_PATH):
    os.remove(PARTIAL_RAW_PATH)

pcam_ds = load_pcam_datasource(sample_frac=SAMPLE_FRAC_PCAM)
pcam_uni = getattr(pcam_ds, "_universe_indices", np.arange(len(pcam_ds), dtype=np.int64))
print("\n[OK] PCam found:", type(pcam_ds).__name__)
print("PCam universe:", len(pcam_uni), "/ full:", len(pcam_ds))

panda_ds = load_panda_datasource(sample_frac=SAMPLE_FRAC_PANDA)
datasets_to_run = [(pcam_ds, pcam_uni)]
if panda_ds is not None:
    panda_uni = getattr(panda_ds, "_universe_indices", np.arange(len(panda_ds), dtype=np.int64))
    print("\n[OK] PANDA found:", type(panda_ds).__name__)
    print("PANDA universe:", len(panda_uni), "/ full:", len(panda_ds))
    datasets_to_run.append((panda_ds, panda_uni))


# -----------------------
# 14.1) PANDA VALIDITY CHECKS (binary proxy + distribution + overlap)
# -----------------------
if panda_ds is not None:
    try:
        y_all = panda_ds.labels_all()[panda_uni]
        print("\n[PANDA VALIDITY] Binary label proxy used in this benchmark.")
        print("[PANDA VALIDITY] Overall counts:", {0: int((y_all==0).sum()), 1: int((y_all==1).sum())})

        splits_check = make_splits_indices(panda_uni, panda_ds.labels_all(), N_SPLITS, VAL_RATIO, SPLIT_BASE_SEED)
        for sid, (trc, vac) in enumerate(splits_check):
            ytr = panda_ds.labels_all()[trc]
            yva = panda_ds.labels_all()[vac]
            inter = set(map(int, trc.tolist())).intersection(set(map(int, vac.tolist())))
            print(
                f"[PANDA VALIDITY] Split {sid}: "
                f"train={len(trc)} (0={int((ytr==0).sum())},1={int((ytr==1).sum())}) | "
                f"val={len(vac)} (0={int((yva==0).sum())},1={int((yva==1).sum())}) | "
                f"overlap={len(inter)}"
            )
    except Exception as e:
        print("[PANDA VALIDITY] check failed:", repr(e))

ALL_METHODS = ["simclr", "byol", "sup_scratch", "sup_imagenet"]

all_raw = []
for ds, uni in datasets_to_run:
    for method in ALL_METHODS:
        df_raw = run_config_resnet18(ds, uni, method=method)
        all_raw.append(df_raw)

results_raw = pd.concat(all_raw, ignore_index=True)

# Summary
group_cols = ["dataset", "method", "backbone", "label_frac", "img_size"]
metric_cols = ["acc", "auc", "f1", "ece", "brier"]

summary = (
    results_raw
    .groupby(group_cols)[metric_cols]
    .agg([np.nanmean, np.nanstd])
    .reset_index()
)
summary.columns = [
    "_".join([c for c in col if c]) if isinstance(col, tuple) else col
    for col in summary.columns
]
for m in metric_cols:
    mu = summary[f"{m}_nanmean"]
    sd = summary[f"{m}_nanstd"]
    summary[f"{m}_mean±std"] = mu.map(lambda v: f"{float(v):.4f}") + " ± " + sd.map(lambda v: f"{float(v):.4f}")

results_summary = summary[group_cols + [f"{m}_mean±std" for m in metric_cols]].copy()

# Save
raw_path = "/kaggle/working/results_raw_resnet.csv"
sum_path = "/kaggle/working/results_summary_resnet.csv"
tex_path = "/kaggle/working/results_summary_resnet.tex"

results_raw.to_csv(raw_path, index=False)
results_summary.to_csv(sum_path, index=False)
latex_df = results_summary.copy()
latex_df["label_frac"] = latex_df["label_frac"].map(lambda v: f"{int(float(v)*100)}\\%")
latex_df.to_latex(tex_path, index=False)

print("\nSaved:")
print(" -", raw_path)
print(" -", sum_path)
print(" -", tex_path)
print(" - partial:", PARTIAL_RAW_PATH)

print("\n=== RESNET RESULTS (mean ± std over 3×3 runs) ===")
try:
    from IPython.display import display
    display(results_summary)
except Exception:
    print(results_summary)
